In [1]:
import numpy as np
import pandas as pd, numpy as np


def walk_forward_splits(n, min_train=1008, test_size=252, embargo=21):
    # expanding-window walk-forward; embargo = purge gap dropped at each train->test boundary
    # train = [0, test_start - embargo), test = [test_start, test_start + test_size)
    # last train label reaches index (test_start-embargo-1)+h; with embargo=h it lands exactly at test_start-1 -> no leak
    test_start = min_train + embargo
    while test_start + test_size <= n:
        train_idx = np.arange(0, test_start - embargo)
        test_idx  = np.arange(test_start, test_start + test_size)
        yield train_idx, test_idx
        test_start += test_size

In [4]:
from google.colab import drive
drive.mount("/content/drive")

PROC = "/content/drive/MyDrive/volatility-forecast/data/processed"  # drive mounted
df = pd.read_csv(f"{PROC}/dataset.csv", index_col=0, parse_dates=True)

feat = ['qqq_ret','hyg_ret','lqd_ret','tlt_ret','gld_ret','vix_lvl','vix_chg',
        'tnx_lvl','tnx_chg','irx_lvl','irx_chg','slope_lvl','slope_chg',
        'credit_lvl','credit_chg','rv1','rv5','rv21']
tgt = ['y_rv1','y_rv5','y_rv21']

work = df.dropna(subset=feat+tgt)   # contiguous middle; shared frame across all h (y_rv21 binds)
print("full:", len(df), "working:", len(work),
      "| head trim:", df.index.get_loc(work.index[0]),
      "| tail trim:", len(df)-1-df.index.get_loc(work.index[-1]))

folds = list(walk_forward_splits(len(work)))
print("folds:", len(folds))
for k,(tr,te) in enumerate(folds):
    gap = te[0]-tr[-1]-1
    print(f"f{k:02d} train[0:{len(tr):4d}] {work.index[0].date()}->{work.index[tr[-1]].date()}"
          f" | gap {gap} | test {work.index[te[0]].date()}->{work.index[te[-1]].date()} n={len(te)}")

Mounted at /content/drive
full: 3873 working: 3831 | head trim: 21 | tail trim: 21
folds: 11
f00 train[0:1008] 2011-02-02->2015-02-04 | gap 21 | test 2015-03-09->2016-03-07 n=252
f01 train[0:1260] 2011-02-02->2016-02-04 | gap 21 | test 2016-03-08->2017-03-07 n=252
f02 train[0:1512] 2011-02-02->2017-02-03 | gap 21 | test 2017-03-08->2018-03-07 n=252
f03 train[0:1764] 2011-02-02->2018-02-05 | gap 21 | test 2018-03-08->2019-03-08 n=252
f04 train[0:2016] 2011-02-02->2019-02-06 | gap 21 | test 2019-03-11->2020-03-09 n=252
f05 train[0:2268] 2011-02-02->2020-02-06 | gap 21 | test 2020-03-10->2021-03-09 n=252
f06 train[0:2520] 2011-02-02->2021-02-05 | gap 21 | test 2021-03-10->2022-03-08 n=252
f07 train[0:2772] 2011-02-02->2022-02-04 | gap 21 | test 2022-03-09->2023-03-09 n=252
f08 train[0:3024] 2011-02-02->2023-02-07 | gap 21 | test 2023-03-10->2024-03-11 n=252
f09 train[0:3276] 2011-02-02->2024-02-08 | gap 21 | test 2024-03-12->2025-03-13 n=252
f10 train[0:3528] 2011-02-02->2025-02-11 | gap 

In [5]:
# persistence: forward-h vol predicted by trailing-h vol (same column family)
pairs = {1:('rv1','y_rv1'), 5:('rv5','y_rv5'), 21:('rv21','y_rv21')}

rows = []
for h,(xcol,ycol) in pairs.items():
    per_fold = []
    for k,(tr,te) in enumerate(folds):
        yhat = work[xcol].values[te]   # prediction = trailing rv_h at t
        ytru = work[ycol].values[te]   # target = forward rv_h
        rmse = np.sqrt(np.mean((yhat-ytru)**2))
        per_fold.append(rmse)
    per_fold = np.array(per_fold)
    rows.append({'h':h,'rmse_mean':per_fold.mean(),'rmse_std':per_fold.std(),
                 'rmse_min':per_fold.min(),'rmse_max':per_fold.max()})
    print(f"h={h:2d} | per-fold RMSE: "+" ".join(f"{v:.4f}" for v in per_fold))

base = pd.DataFrame(rows).set_index('h')
print()
print(base.round(4))

h= 1 | per-fold RMSE: 0.0113 0.0066 0.0077 0.0126 0.0107 0.0177 0.0113 0.0169 0.0095 0.0113 0.0131
h= 5 | per-fold RMSE: 0.0063 0.0045 0.0050 0.0065 0.0067 0.0104 0.0044 0.0079 0.0044 0.0047 0.0091
h=21 | per-fold RMSE: 0.0053 0.0037 0.0039 0.0060 0.0116 0.0103 0.0040 0.0046 0.0021 0.0047 0.0080

    rmse_mean  rmse_std  rmse_min  rmse_max
h                                          
1      0.0117    0.0032    0.0066    0.0177
5      0.0064    0.0019    0.0044    0.0104
21     0.0058    0.0028    0.0021    0.0116
